<a href="https://colab.research.google.com/github/Diishasing/MAXIMUM_REVENUE-Statistical-project/blob/main/max_revenue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ***MAXIMUM REVENUE FOR THE DRIVERS***




#AGENDA
1. Problem statement
2. Research question
3. Data overview
4. Methodology: descriptive analysis, hypothesis testing and regression analysis(what would be the fare_amount for trip_duration)
5. Journey insights[all graphs and bar charts]
6. Recommendations

**PROBLEM STATEMENT:**

---


In the fast-paced taxi booking sector, making the most of revenue is essential for long-term success and driver happiness. Our goal is to use data-driven insights to maximise revenue streams for taxi drivers in order to meet this need. Our research aims to determine whether payment methods have an impact on fare pricing by focusing on the relationship between payment type and fare amount.

**RESEARCH QUESTION:**

---
Is there a relationship between total fare amount and payment type?
Can we nudge customers towards payment methods that generate higher revenue for drivers, without negatively impacting customer experience?


IMPORTING LIBRARIES

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import warnings
import statsmodels.api as sm

warnings.filterwarnings('ignore')

LOADING DATASETS

**DATA OVERVIEW:**

---

For this analysis, we utilized the comprehensive dataset of NYC Taxi Trip records, used data cleaning and feature engineering procedures to concentrate solely on the relevant columns essential for our investigation.

In [ ]:
df = pd.read_csv('data.csv')
print(df.shape)
df.head()

EXPLORATORY DATA ANALYSIS

In [ ]:
df.dtypes

In [ ]:
#change date column to datetime
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

In [ ]:
df['duration'] = df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']

In [ ]:
#coverting duration to minutes
df['duration'] = df['duration'].dt.total_seconds()/60
df.head()

RELEVANT COLUMNS USED FOR THIS PROBLEM STATMENT:

---


*   passenger_count (1 to 5)
*   payment_type (card or cash)
*   fare_amount
*   trip_distance (miles)
*   duration (minutes)




In [ ]:
df = df[['passenger_count', 'payment_type', 'fare_amount', 'trip_distance', 'duration']]

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace = True)

In [ ]:
df['passenger_count'] = df['passenger_count'].astype('int64')
df['payment_type'] = df['payment_type'].astype('int64')

In [ ]:
df[df.duplicated()]

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df['passenger_count'].value_counts(normalize=True)

In [ ]:
df['payment_type'].value_counts(normalize=True)


In [ ]:
df = df[df['payment_type']<3]
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] < 6)]

In [ ]:
df['payment_type'] = df['payment_type'].map({1:'Card', 2: 'Cash'})

In [ ]:
df.describe()

In [ ]:
df = df[df['fare_amount']>0]
df = df[df['trip_distance']>0]
df = df[df['duration']>0]

In [ ]:
plt.hist(df['fare_amount'])
plt.boxplot(df['fare_amount'])

In [ ]:
for col in ['fare_amount', 'trip_distance', 'duration']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df = df[(df[col] > lower_bound) & (df[col] < upper_bound)]

In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.title('Distribution of Fare Amount')
plt.hist(df[df['payment_type'] == 'Card']['fare_amount'], histtype='barstacked', bins = 20, edgecolor = 'k', color = '#FA643F')
plt.hist(df[df['payment_type'] == 'Cash']['fare_amount'], histtype='barstacked', bins = 20, edgecolor = 'k', color = '#FFBCAB')
plt.legend(['Card', 'Cash'])
plt.show()

**JOURNEY INSIGHTS #1:**

---
Customers paying with cards tend to have a slightly higher average trip distance and fare amount compared to those paying the cash.


In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.title('Distribution of Trip Distance')
plt.hist(df[df['payment_type'] == 'Card']['trip_distance'], histtype='barstacked', bins = 20, edgecolor = 'k', color = '#FA643F')
plt.hist(df[df['payment_type'] == 'Cash']['trip_distance'], histtype='barstacked', bins = 20, edgecolor = 'k', color = '#FFBCAB')
plt.legend(['Card', 'Cash'])
plt.show()

**JOURNEY INSIGHTS #2:**

---

Indicates that customer prefers to pay more with cards when they have high fare amount and long trip distance.

In [ ]:
df.groupby('payment_type').agg({'fare_amount' : ['mean', 'std'],
                                'trip_distance' : ['mean', 'std']})

In [ ]:
plt.title('Preference of Payment Type')
plt.pie(df['payment_type'].value_counts(normalize = True), labels = df['payment_type'].value_counts().index,
        startangle= 90, shadow = True, autopct = '%1.1f%%', colors = ['#FA64F3', '#FFBCAB'])
plt.show()

**JOURNEY INSIGHTS #3:**

---
This indicates a strong preferences among customers for using card payments over cash, potentially due to convenience, security, or incentives offered for card transactions.


In [ ]:
passenger_count = df.groupby(['payment_type', 'passenger_count'])[['passenger_count']].count()
passenger_count.rename(columns = {'passenger_count':'count'}, inplace = True)
passenger_count.reset_index(inplace = True)

In [ ]:
passenger_count['percent'] = (passenger_count['count']/passenger_count['count'].sum())

In [ ]:
passenger_count

In [ ]:
rf = pd.DataFrame(columns = ['payment_type', 1, 2, 3, 4, 5])
rf['payment_type'] = ['Card', 'Cash']
rf.iloc[0, 1:] = passenger_count.iloc[0:5, -1]
rf.iloc[1, 1:] = passenger_count.iloc[5:, -1]
rf

**PASSENGER COUNT ANALYSIS:**

---

* Among card payments, rides with a single passenger (passenger_count = 1) comprise the largest proportion, constituting 40.08% of all card transactions.
* Similarly, cash payments are predominantly associated with single-passenger ride, making up 20.04% of all cash transactions
* There is a noticeable decrease in the percentage of transactions as the passenger count increases, suggesting that larger groups are less likely to use taxis or may opt for alternative payments methods.
* These insights emphasize the importance of considering both payment method and passenger count when analyzing transaction data, as they provide valuable insights into customer behavior and preferences.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 6))
rf.plot(x = 'payment_type', kind ='barh', stacked=True, color=['#FA643F', '#FFBCAB', '#CBB2B2', '#F1F1F1', '#FD9F9F'], ax=ax)

#ADD PERCENTAGES AND TEXTS
for p in ax.patches:
  width = p.get_width()
  height = p.get_height()
  x, y = p.get_xy()
  ax.text(x+width/2,
          y+height/2,
          '{:.0%}'.format(width),
          horizontalalignment='center',
          verticalalignment='center')

plt.show()

**NULL HYPOTHESIS:**

---


There is no difference in average fare between customers who use credit cards and customers who use cash

**ALTERNATE HYPOTHESIS:**

---


There is a difference in average fare between customers who use credit cards and customers who use cash


In [ ]:
sm.qqplot(df['fare_amount'], line = '45')
plt.show()

In [ ]:
card_sample = df[df['payment_type']=='Card']['fare_amount']
cash_sample = df[df['payment_type']=='Cash']['fare_amount']


Performing T-statistic and finding the p-value

In [ ]:
t_stat, p_value = st.ttest_ind(a = card_sample, b = cash_sample, equal_var = False)
print('T-statistics', t_test, 'p_value', p_value)

With a T-statistic of 165.5 and a P-value of less than 0.05, we reject the null hypothesis, suggesting that there is indeed a significant difference in average fare between the two payment methods.




Hence, the alternate hypothesis is correct since the p-value is less than 0.05

**RECOMMENDATIONS:**

---



*   Implement strategies such as offering incentives or discounts for credit card transactions to incentivize customers to choose this payment method.
*   Provide seamless and secure credit card payment options to enhance customer convenience and encourage adoption of this preferred payment method.
*   Encourage customers to pay with credit cards to capitalize on the potential for generating more revenue for taxi can drivers.



# **THANK YOU!** :>